In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# AI transpiler passes

AI transpiler passes คือ passes ที่ทำหน้าที่แทนที่ passes แบบ "ดั้งเดิม" ของ Qiskit สำหรับงาน transpiling บางอย่าง โดยมักให้ผลลัพธ์ที่ดีกว่าอัลกอริทึมฮิวริสติกที่มีอยู่ (เช่น depth ต่ำกว่าและจำนวน CNOT น้อยกว่า) และยังเร็วกว่าอัลกอริทึมการปรับแต่งอย่าง Boolean satisfiability solvers มาก AI transpiler passes รันได้ในสภาพแวดล้อมเครื่องของตัวเอง

> **Note:** AI transpiler passes อยู่ในสถานะ beta release ซึ่งอาจมีการเปลี่ยนแปลง
>     หากมีข้อเสนอแนะหรือต้องการติดต่อทีมพัฒนา สามารถใช้ [Qiskit Slack Workspace channel](https://qiskit.slack.com/archives/C06KF8YHUAU) นี้ได้เลย

passes ที่ใช้งานได้ในปัจจุบันมีดังนี้:

**Routing passes**
 - `AIRouting`: การเลือก layout และการ routing ของ Circuit

**Circuit synthesis passes**
 - `AICliffordSynthesis`: การ synthesis ของ Clifford Circuit
 - `AILinearFunctionSynthesis`: การ synthesis ของ Linear Function Circuit
 - `AIPermutationSynthesis`: การ synthesis ของ Permutation Circuit

ในการใช้งาน AI transpiler passes ต้องติดตั้ง package `qiskit-ibm-transpiler` ก่อน และสามารถดูข้อมูลเพิ่มเติมเกี่ยวกับตัวเลือกต่าง ๆ ได้ที่ [qiskit-ibm-transpiler API documentation](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler)

## รัน AI transpiler passes บนเครื่องหรือบนคลาวด์
ติดตั้ง `qiskit-ibm-transpiler` พร้อม dependencies เพิ่มเติมดังนี้:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService
import logging

backend = QiskitRuntimeService().backend("ibm_fez")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)
transpiled_circuit = ai_passmanager.run(circuit)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

เมื่อติดตั้ง dependencies เพิ่มเติมแล้ว โหมดเริ่มต้นในการรัน AI transpiler passes จะใช้เครื่องของตัวเอง

## AI routing pass
`AIRouting` pass ทำหน้าที่ทั้งเป็น layout stage และ routing stage สามารถใช้ภายใน `PassManager` ได้ดังนี้:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit.circuit.library import efficient_su2

ibm_kingston = QiskitRuntimeService().backend("ibm_kingston")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_kingston,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_kingston, local_mode=True
        ),  # Re-synthesize Linear Function blocks
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Fetching 127 files:   0%|          | 0/127 [00:00<?, ?it/s]

The synthesis respects the coupling map of the device: it can be run safely after other routing passes without disturbing the circuit, so the overall circuit will still follow the device restrictions. By default, the synthesis will replace the original sub-circuit only if the synthesized sub-circuit improves the original (currently only checking CNOT count), but this can be forced to always replace the circuit by setting `replace_only_if_better=False`.

The following synthesis passes are available from `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Synthesis for [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) circuits (blocks of `H`, `S`, and `CX` gates). Currently up to nine qubit blocks.
- *AILinearFunctionSynthesis*: Synthesis for [Linear Function](/docs/api/qiskit/qiskit.circuit.library.LinearFunction) circuits (blocks of `CX` and `SWAP` gates). Currently up to nine qubit blocks.
- *AIPermutationSynthesis*: Synthesis for [Permutation](/docs/api/qiskit/qiskit.circuit.library.Permutation#permutation) circuits (blocks of `SWAP` gates). Currently available for 65, 33, and 27 qubit blocks.
- *AIPauliNetworkSynthesis*: Synthesis for Pauli Network circuits (blocks of `H`, `S`, `SX`, `CX`, `RX`, `RY` and `RZ` gates). Currently up to six qubit blocks.

We expect to gradually increase the size of the supported blocks.

All passes use a thread pool to send several requests in parallel. By default, the number for max threads is the number of cores plus four (default values for the `ThreadPoolExecutor` Python object). However, you can set your own value with the `max_threads` argument at pass instantiation. For example, the following line instantiates the `AILinearFunctionSynthesis` pass, which allows it to use a maximum of 20 threads.

```python
AILinearFunctionSynthesis(backend=ibm_torino, max_threads=20)  # Re-synthesize Linear Function blocks using 20 threads max
```

You can also set the environment variable `AI_TRANSPILER_MAX_THREADS` to the desired number of maximum threads, and all synthesis passes instantiated after that will use that value.

For the AI synthesis passes to synthesize a sub-circuit, it must lay on a connected subgraph of the coupling map (one way to do this is with a routing pass before collecting the blocks, but this is not the only way to do it). The synthesis passes will automatically check that the specific subgraph is supported, and if not, it will raise a warning and leave the original sub-circuit unchanged.

The following custom collection passes for Cliffords, Linear Functions and Permutations that can be imported from `qiskit_ibm_transpiler.ai.collection` also complement the synthesis passes:

- *CollectCliffords*: Collects Clifford blocks as `Instruction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectLinearFunctions*: Collects blocks of `SWAP` and `CX` as `LinearFunction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectPermutations*: Collects blocks of `SWAP` circuits as `Permutations`.
- *CollectPauliNetworks*: Collects Pauli Network blocks and stores the original sub-circuit to compare against it after synthesis.

These custom collection passes limit the sizes of the collected sub-circuits so they are supported by the AI-powered synthesis passes. Therefore, it is recommended to use them after the routing passes and before the synthesis passes for a better overall optimization.

## Hybrid heuristic-AI circuit transpilation

The `qiskit-ibm-transpiler` allows you to configure a hybrid pass manager that combines the best of Qiskit's heuristic and the AI-powered transpiler passes. This feature behaves similarly to the Qiskit `generate_pass_manager` method. A typical way to use `generate_ai_pass_manager` is as follows:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_kingston")
kingston_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=kingston_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

ในที่นี้ `backend` กำหนด coupling map ที่จะใช้ routing, `optimization_level` (1, 2 หรือ 3) กำหนดความพยายามในการคำนวณที่จะใช้ในกระบวนการ (ค่าสูงกว่ามักให้ผลดีกว่าแต่ใช้เวลานานกว่า) และ `layout_mode` กำหนดวิธีจัดการการเลือก layout
`layout_mode` มีตัวเลือกดังนี้:

- `keep`: รักษา layout ที่กำหนดโดย transpiler passes ก่อนหน้า (หรือใช้ trivial layout หากไม่ได้กำหนด) มักใช้เฉพาะเมื่อ Circuit ต้องรันบน qubits เฉพาะของอุปกรณ์ มักให้ผลลัพธ์ที่แย่กว่าเพราะมีพื้นที่สำหรับการปรับแต่งน้อยกว่า
- `improve`: ใช้ layout ที่กำหนดโดย transpiler passes ก่อนหน้าเป็นจุดเริ่มต้น เหมาะเมื่อมีการเดา layout เริ่มต้นที่ดี เช่น Circuit ที่สร้างขึ้นในลักษณะที่ใกล้เคียงกับ coupling map ของอุปกรณ์ ยังเหมาะหากต้องการลอง layout passes อื่น ๆ ร่วมกับ `AIRouting` pass
- `optimize`: นี่คือโหมดเริ่มต้น ทำงานได้ดีที่สุดสำหรับ Circuit ทั่วไปที่อาจไม่มีการเดา layout ที่ดี โหมดนี้จะละเว้นการเลือก layout ก่อนหน้า
- `local_mode`: flag นี้กำหนดว่า `AIRouting` pass จะรันที่ไหน ถ้าเป็น `False`, `AIRouting` จะรันจากระยะไกลผ่าน Qiskit Transpiler Service ถ้าเป็น `True` package จะพยายามรัน pass ในสภาพแวดล้อมเครื่องของตัวเอง โดยถ้าไม่พบ dependencies ที่จำเป็นจะ fallback ไปใช้โหมดคลาวด์

## AI circuit synthesis passes
AI circuit synthesis passes ช่วยให้ปรับแต่งส่วนต่าง ๆ ของ Circuit ประเภทต่าง ๆ ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation)) ด้วยการ re-synthesize วิธีทั่วไปในการใช้ synthesis pass มีดังนี้: